# Aligned Advisor — LLM-Judge Prompt Optimiser

Iteratively aligns a judge prompt with human-expert golden labels from
`advisor_golden_test_cases.xlsx`.

**Flow**
1. Start with `prompt_0` — a judge that evaluates the structured JSON extraction produced by the Advisor agent.
2. For each round, send every test-case row to the webhook with `{"input": filled_prompt}`.
3. Parse the JSON response; compare predicted labels against golden labels.
4. If **all** fields have mismatch rate ≤ 30 % → stop.
5. Otherwise feed mismatched rows into `OptPrompt` to obtain an improved prompt, then repeat.
6. Stop after **5 rounds** at most.

### What is evaluated?
The Advisor agent receives a Thai debt-restructuring conversation and extracts 8 fields:
- `consultAcc` — comma-separated matched account numbers
- `DebtSituation` — debt situation category
- `maxPayment` — max monthly payment (THB)
- `maxTerm` — payoff term (months)
- `refPlanID` — referenced plan ID
- `maxPaymentY2` — Year 2 payment cap
- `maxPaymentY3` — Year 3 payment cap
- `reConfirmMessage` — CLARIFY template name or `""` (EXTRACT mode)

**Score fields:** all 8 above → golden columns: `rating_<field>`  
**Allowed values:** `"good"` / `"acceptable"` / `"invalid"`

In [ ]:
import os
import re
import json
import time
import requests
import pandas as pd
from tqdm.notebook import tqdm

# ── Configuration ─────────────────────────────────────────────────────────────
N8N_BASE_URL       = "https://alphamakeathon-automation.arisetech.dev"
WEBHOOK_PATH       = "9d6049fc-77c8-43e9-b71e-f734506f4f9d"   # ← fill in
USE_TEST_URL       = False   # True → /webhook-test/…  |  False → /webhook/…

GOLDEN_FILE        = "golden_test_cases/advisor_golden_test_cases.xlsx"
RESULT_DIR         = "advisor_test_result"

MAX_ROUNDS         = 5
MISMATCH_THRESHOLD = 0.30    # stop when ALL fields are within this rate
TIMEOUT            = 600      # seconds per request (10 minutes)
RETRIES            = 2
DELAY_BETWEEN      = 0.5     # seconds between rows

SCORE_FIELDS = [
    "consultAcc", "DebtSituation", "maxPayment", "maxTerm",
    "refPlanID", "maxPaymentY2", "maxPaymentY3", "reConfirmMessage",
]


def golden_col(f: str) -> str:
    """Map a score-field name to its golden-label column in the DataFrame."""
    return f"rating_{f}"


def get_webhook_url() -> str:
    prefix = "webhook-test" if USE_TEST_URL else "webhook"
    return f"{N8N_BASE_URL}/{prefix}/{WEBHOOK_PATH}"


print("Webhook URL:", get_webhook_url())

## OptPrompt

Meta-prompt sent to the webhook when mismatches exceed the threshold.
Placeholders: **`{oldPrompt}`**, **`{mismatchResults}`**.
The LLM must return an *UpdatePrompt* whose own placeholders cover all conversation-context
fields and extracted result fields listed below.

In [ ]:
OptPrompt = (
    "You are an expert prompt engineer specialising in LLM-as-a-judge systems "
    "for evaluating AI-generated structured extraction outputs at a Thai commercial bank.\n\n"
    "Your task is to improve the judge prompt below so it better aligns with "
    "human-expert (golden) evaluations.\n\n"
    "## Current Prompt\n"
    "{oldPrompt}\n\n"
    "## Mismatch Cases\n"
    "The following rows show where the current prompt disagreed with golden labels. "
    "Each entry includes the conversation context, extracted fields, "
    "the golden label, and the predicted label:\n"
    "{mismatchResults}\n\n"
    "## Your Task\n"
    "1. Analyse the mismatch patterns.\n"
    "2. Identify what the current prompt misinterprets.\n"
    "3. Rewrite the prompt to fix those issues while preserving correct behaviours.\n\n"
    "## Requirements for the output prompt\n"
    "- MUST keep exactly these placeholders (with single curly braces):\n"
    "  Conversation context: {userMessage}, {LastAImessage}, {accText}, {offerText},\n"
    "  {consultAcc}, {DebtSituation}, {maxPayment}, {maxTerm}, {refPlanID},\n"
    "  {maxPaymentY2}, {maxPaymentY3}, {narrative}\n"
    "  Extracted results: {result_consultAcc}, {result_DebtSituation}, {result_maxPayment},\n"
    "  {result_maxTerm}, {result_refPlanID}, {result_maxPaymentY2}, {result_maxPaymentY3},\n"
    "  {result_reConfirmMessage}\n"
    "- MUST instruct the LLM to return ONLY a JSON object with these fields and allowed values:\n"
    '    { "consultAcc":       "good"|"acceptable"|"invalid",\n'
    '      "DebtSituation":    "good"|"acceptable"|"invalid",\n'
    '      "maxPayment":       "good"|"acceptable"|"invalid",\n'
    '      "maxTerm":          "good"|"acceptable"|"invalid",\n'
    '      "refPlanID":        "good"|"acceptable"|"invalid",\n'
    '      "maxPaymentY2":     "good"|"acceptable"|"invalid",\n'
    '      "maxPaymentY3":     "good"|"acceptable"|"invalid",\n'
    '      "reConfirmMessage": "good"|"acceptable"|"invalid" }\n\n'
    "Return ONLY the improved prompt text — no preamble, no explanation."
)

print("OptPrompt defined:", len(OptPrompt), "chars")

## Initial Judge Prompt — `prompt_0`

Evaluates the correctness of each field extracted by the Advisor agent from a Thai debt-restructuring conversation.

Placeholders: all conversation-context fields plus
**`{result_consultAcc}`**, **`{result_DebtSituation}`**, **`{result_maxPayment}`**,
**`{result_maxTerm}`**, **`{result_refPlanID}`**, **`{result_maxPaymentY2}`**,
**`{result_maxPaymentY3}`**, **`{result_reConfirmMessage}`**.

Returns JSON with `consultAcc`, `DebtSituation`, `maxPayment`, `maxTerm`,
`refPlanID`, `maxPaymentY2`, `maxPaymentY3`, `reConfirmMessage`,
each valued **`"good"`** / **`"acceptable"`** / **`"invalid"`**.

In [ ]:
prompt_0 = (
    "You are a Quality Assurance (QA) expert evaluating the structured extraction output "
    "produced by an AI debt-restructuring information extraction agent for KTB Care.\n\n"
    "The agent reads a Thai bank customer conversation and extracts structured fields "
    "that staff will use to process a debt restructuring request. "
    "Your task is to evaluate whether each extracted field is "
    '"good" (correct), '
    '"acceptable" (minor deviation, still usable), or '
    '"invalid" (wrong or misleading).\n\n'
    "## Agent Operating Rules (summary)\n\n"
    "**EXTRACT mode** (`reConfirmMessage = \"\"`): used when the customer provides new "
    "information — concrete numeric values, account/product references, situation description, "
    "or no plan has been offered yet.\n\n"
    "**CLARIFY mode** (`reConfirmMessage = <template>`): used only when (a) plans have been "
    "offered, (b) the customer references an offered plan, and (c) the customer expresses "
    "vague dissatisfaction without giving a concrete number. Templates:\n"
    "  CLARIFY_HIGH_INTEREST — total interest too high\n"
    "  CLARIFY_HIGH_BALLOON — balloon/closing payment too high\n"
    "  CLARIFY_HIGH_INSTALLMENT — monthly installment too high, no number given\n"
    "  CLARIFY_LONG_TERM — term too long, no number given\n"
    "  CLARIFY_HIGH_Y2Y3 — Year 2/3 step-up payments too high\n\n"
    "**Persistence rule**: carry forward all previous values for fields the customer "
    "does not explicitly update.\n\n"
    "## Conversation Context\n\n"
    "Latest Customer Message:\n{userMessage}\n\n"
    "Latest Assistant Message:\n{LastAImessage}\n\n"
    "Customer Accounts (accText):\n{accText}\n\n"
    "Previously Offered Solutions (offerText):\n{offerText}\n\n"
    "Conversation Narrative:\n{narrative}\n\n"
    "## Previous Field Values (carried in from prior turns)\n\n"
    "Previously Selected Accounts: {consultAcc}\n"
    "Previously Extracted Debt Situation: {DebtSituation}\n"
    "Previous Maximum Monthly Payment: {maxPayment}\n"
    "Previous Desired Payoff Term (months): {maxTerm}\n"
    "Previous Referenced Plan ID: {refPlanID}\n"
    "Previous Max Payment Year 2: {maxPaymentY2}\n"
    "Previous Max Payment Year 3: {maxPaymentY3}\n\n"
    "## Extracted Result to Evaluate\n\n"
    "consultAcc:        {result_consultAcc}\n"
    "DebtSituation:     {result_DebtSituation}\n"
    "maxPayment:        {result_maxPayment}\n"
    "maxTerm:           {result_maxTerm}\n"
    "refPlanID:         {result_refPlanID}\n"
    "maxPaymentY2:      {result_maxPaymentY2}\n"
    "maxPaymentY3:      {result_maxPaymentY3}\n"
    "reConfirmMessage:  {result_reConfirmMessage}\n\n"
    "## Evaluation Criteria\n\n"
    "### consultAcc — Comma-separated matched account numbers\n"
    '- "good": Exactly matches the accounts the customer referenced. '
    "Includes all accounts when 'ทั้งหมด'/'ทุกบัญชี' is stated or when the assistant "
    "listed/confirmed specific accounts. Preserves previous value when no account "
    "is mentioned. Output has no spaces between commas.\n"
    '- "acceptable": Correct accounts identified but minor formatting issue '
    "(extra spaces, different order) or one account ambiguously included/excluded "
    "where the reference was unclear.\n"
    '- "invalid": Wrong accounts included or excluded, or previous value not preserved '
    "when no new account was mentioned.\n\n"
    "### DebtSituation — Debt situation category\n"
    "Categories: DebtBurden | TemporaryCashflow | PermanentAffordability | CareerChange | FinancialShock\n"
    '- "good": Correctly classifies the customer\'s debt situation based on what they stated.\n'
    '- "acceptable": Close category but slightly imprecise (e.g. DebtBurden vs TemporaryCashflow '
    "when either could apply); preserves previous value correctly when no new information given.\n"
    '- "invalid": Wrong category, or value changed without the customer providing new situation info.\n\n'
    "### maxPayment — Maximum monthly payment (THB)\n"
    '- "good": Correct numeric value extracted (strips symbols, converts Thai numerals); '
    "carries forward previous value when not mentioned; uses accepted plan's installment when customer accepts.\n"
    '- "acceptable": Off by a small rounding difference or reasonable interpretation of an ambiguous amount.\n'
    '- "invalid": Wrong value, zero when a value was stated, or previous value not carried forward.\n\n'
    "### maxTerm — Payoff term in months\n"
    '- "good": Correct value (years converted to months: 3 ปี → 36, 5 ปี → 60); '
    "previous value carried forward when not mentioned.\n"
    '- "acceptable": Minor conversion ambiguity but intent is clear.\n'
    '- "invalid": Wrong value or previous value not carried forward.\n\n'
    "### refPlanID — Referenced plan ID from offered solutions\n"
    '- "good": Correct planID when customer references a specific plan; null when customer '
    "does not reference any existing plan.\n"
    '- "acceptable": Reasonable planID chosen when reference was ambiguous '
    "(e.g. multiple plans offered, customer says 'แผนที่เสนอ').\n"
    '- "invalid": Wrong planID, or non-null when customer clearly did not reference any plan.\n\n'
    "### maxPaymentY2 / maxPaymentY3 — Year 2 / Year 3 payment caps\n"
    '- "good": Null when customer never mentioned Y2/Y3; correct value when explicitly stated.\n'
    '- "acceptable": Minor rounding or reasonable interpretation of a vague Y2/Y3 statement.\n'
    '- "invalid": Non-null value fabricated when customer said nothing about Y2/Y3, or '
    "wrong value when they did.\n\n"
    "### reConfirmMessage — CLARIFY template or empty string\n"
    '- "good": Correct template name in CLARIFY mode (all conditions met); '
    "empty string in EXTRACT mode.\n"
    '- "acceptable": Broadly correct mode but template name is the closest alternative '
    "when the exact trigger phrase is ambiguous.\n"
    '- "invalid": Wrong mode (CLARIFY when should be EXTRACT or vice versa), '
    "or wrong template name.\n\n"
    "## Output Instructions\n"
    "Return ONLY a JSON object — no markdown, no explanation:\n"
    "{\n"
    '  "consultAcc":       "good" | "acceptable" | "invalid",\n'
    '  "DebtSituation":    "good" | "acceptable" | "invalid",\n'
    '  "maxPayment":       "good" | "acceptable" | "invalid",\n'
    '  "maxTerm":          "good" | "acceptable" | "invalid",\n'
    '  "refPlanID":        "good" | "acceptable" | "invalid",\n'
    '  "maxPaymentY2":     "good" | "acceptable" | "invalid",\n'
    '  "maxPaymentY3":     "good" | "acceptable" | "invalid",\n'
    '  "reConfirmMessage": "good" | "acceptable" | "invalid"\n'
    "}"
)

print("prompt_0 defined:", len(prompt_0), "chars")

## Helper Functions

In [ ]:
_PROMPT_FIELDS = [
    # conversation context
    "userMessage", "LastAImessage", "accText", "offerText", "narrative",
    # previous field values
    "consultAcc", "DebtSituation", "maxPayment", "maxTerm",
    "refPlanID", "maxPaymentY2", "maxPaymentY3",
    # extracted results
    "result_consultAcc", "result_DebtSituation", "result_maxPayment",
    "result_maxTerm", "result_refPlanID", "result_maxPaymentY2",
    "result_maxPaymentY3", "result_reConfirmMessage",
]


def fill_prompt(template: str, row: pd.Series) -> str:
    filled = template
    for col in _PROMPT_FIELDS:
        filled = filled.replace(f"{{{col}}}", str(row.get(col, "")))
    return filled


def fill_opt_prompt(template: str, oldPrompt: str, mismatchResults: str) -> str:
    return (template
            .replace("{oldPrompt}",       str(oldPrompt))
            .replace("{mismatchResults}", str(mismatchResults)))


def _call_raw(payload: dict, timeout: int = TIMEOUT, retries: int = RETRIES):
    url = get_webhook_url()
    last_exc: Exception | None = None
    for attempt in range(retries + 1):
        try:
            resp = requests.post(url, json=payload, timeout=timeout)
            resp.raise_for_status()
            return resp.json()
        except requests.exceptions.RequestException as exc:
            last_exc = exc
            if attempt < retries:
                time.sleep(1.5 * (attempt + 1))
    raise last_exc


def call_webhook(prompt: str):
    return _call_raw({"input": prompt})


def get_response_text(resp) -> str:
    if isinstance(resp, str):
        return resp
    if isinstance(resp, dict):
        for key in ("output", "text", "result", "response", "content", "message"):
            if key in resp and resp[key] is not None:
                return str(resp[key])
        return json.dumps(resp, ensure_ascii=False)
    return str(resp)


def parse_json_response(resp) -> dict:
    text = get_response_text(resp)
    # 1. markdown code block
    m = re.search(r"```(?:json)?\s*([\s\S]*?)\s*```", text)
    if m:
        try:
            return json.loads(m.group(1))
        except json.JSONDecodeError:
            pass
    # 2. whole text as JSON
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    # 3. first JSON object found
    m = re.search(r"(\{[\s\S]*?\})", text)
    if m:
        try:
            return json.loads(m.group(1))
        except json.JSONDecodeError:
            pass
    return {}


def extract_prompt_text(resp) -> str:
    text = get_response_text(resp)
    m = re.search(r"```(?:\w+)?\s*([\s\S]*?)\s*```", text)
    if m:
        return m.group(1).strip()
    return text.strip()


print("Helpers ready.")

## Load Golden Test Cases

In [ ]:
df_golden = pd.read_excel(GOLDEN_FILE)

# Normalise labels to lowercase so they match the judge's output values
for f in SCORE_FIELDS:
    col = golden_col(f)
    if col in df_golden.columns:
        df_golden[col] = df_golden[col].astype(str).str.strip().str.lower()
        df_golden[col] = df_golden[col].replace("nan", pd.NA)

print(f"Loaded {len(df_golden)} golden test cases")
print("Columns:", df_golden.columns.tolist())
print("\nLabel distributions:")
for f in SCORE_FIELDS:
    col = golden_col(f)
    if col in df_golden.columns:
        print(f"  {col}: {df_golden[col].value_counts().to_dict()}")
df_golden.head(3)

## Optimisation Loop

For each round:
1. Fill all conversation-context and extracted-result placeholders in the current judge prompt.
2. POST `{"input": filled_prompt}` to the webhook; collect responses.
3. Parse each response string → dict with `re` / `json`.
4. Save `test_result_prompt_N.xlsx`.
5. Compute per-field mismatch rate vs golden labels (`rating_*` columns, normalised to lowercase).
6. **Stop** if every field's rate ≤ 30 %, or after 5 rounds.
7. Otherwise collect mismatch rows → fill `OptPrompt` → POST to webhook → use response as next prompt.

In [ ]:
os.makedirs(RESULT_DIR, exist_ok=True)

current_prompt = prompt_0
all_rounds: list[dict] = []

for round_idx in range(MAX_ROUNDS):
    round_name  = f"prompt_{round_idx}"
    out_path    = os.path.join(RESULT_DIR, f"test_result_{round_name}.xlsx")
    prompt_path = os.path.join(RESULT_DIR, f"{round_name}.txt")

    print(f"\n{'=' * 60}")
    print(f"Round {round_idx}  |  {round_name}")
    print(f"{'=' * 60}")

    # ── Save prompt as text file ───────────────────────────────────────────
    with open(prompt_path, "w", encoding="utf-8") as fh:
        fh.write(current_prompt)
    print(f"  Saved prompt → {prompt_path}")

    # ── Check if result xlsx already exists — skip evaluation if so ───────
    if os.path.exists(out_path):
        print(f"  Result already exists → {out_path}  (skipping evaluation)")
        df_test_result = pd.read_excel(out_path)
        df_pred = pd.DataFrame({f: df_test_result[f"pred_{f}"].values for f in SCORE_FIELDS})
    else:
        # ── 1. Run evaluation ──────────────────────────────────────────────
        predicted: list[dict] = []
        errors:    list[str | None] = []

        for _, row in tqdm(df_golden.iterrows(), total=len(df_golden), desc=round_name):
            try:
                filled = fill_prompt(current_prompt, row)
                raw    = call_webhook(filled)
                parsed = parse_json_response(raw)
                predicted.append({f: parsed.get(f) for f in SCORE_FIELDS})
                errors.append(None)
            except Exception as exc:
                predicted.append({f: None for f in SCORE_FIELDS})
                errors.append(str(exc))
            time.sleep(DELAY_BETWEEN)

        df_pred = pd.DataFrame(predicted)

        # ── 2. Build test_result dataframe ────────────────────────────────
        df_test_result = df_golden[["testId", "userMessage"]].copy()
        for f in SCORE_FIELDS:
            df_test_result[golden_col(f)]  = df_golden[golden_col(f)].values
            df_test_result[f"pred_{f}"]    = df_pred[f].values
            df_test_result[f"match_{f}"]   = (df_pred[f].values == df_golden[golden_col(f)].values)
        df_test_result["error"] = errors

        df_test_result.to_excel(out_path, index=False)
        print(f"  Saved → {out_path}")

    # ── 3. Compute per-field mismatch rates ───────────────────────────────
    mismatch_rates: dict[str, float] = {}
    for f in SCORE_FIELDS:
        valid = df_golden[golden_col(f)].notna() & df_pred[f].notna()
        if valid.sum():
            mismatch_rates[f] = float((~df_test_result[f"match_{f}"][valid]).mean())
        else:
            mismatch_rates[f] = float("nan")

    print("\n  Mismatch rates:")
    for f, rate in mismatch_rates.items():
        tag = "✓" if (not pd.isna(rate) and rate <= MISMATCH_THRESHOLD) else "✗"
        print(f"    {f:25s}: {rate:.1%}  {tag}")

    all_rounds.append({
        "round":          round_idx,
        "prompt_name":    round_name,
        "prompt":         current_prompt,
        "df_test_result": df_test_result.copy(),
        "mismatch_rates": mismatch_rates.copy(),
    })

    # ── 4. Stopping condition ─────────────────────────────────────────────
    valid_rates  = [v for v in mismatch_rates.values() if not pd.isna(v)]
    max_mismatch = max(valid_rates) if valid_rates else float("nan")

    if not pd.isna(max_mismatch) and max_mismatch <= MISMATCH_THRESHOLD:
        print(f"\n  All fields within {MISMATCH_THRESHOLD:.0%} threshold. Stopping.")
        break

    if round_idx == MAX_ROUNDS - 1:
        print(f"\n  Reached maximum {MAX_ROUNDS} rounds. Stopping.")
        break

    # ── 5. Collect mismatched rows ─────────────────────────────────────────
    mismatch_mask = pd.Series(False, index=df_golden.index)
    for f in SCORE_FIELDS:
        mismatch_mask |= (df_pred[f].values != df_golden[golden_col(f)].values)

    context_cols = [
        "testId", "userMessage", "LastAImessage", "accText", "offerText", "narrative",
        "consultAcc", "DebtSituation", "maxPayment", "maxTerm",
        "refPlanID", "maxPaymentY2", "maxPaymentY3",
        "result_consultAcc", "result_DebtSituation", "result_maxPayment",
        "result_maxTerm", "result_refPlanID", "result_maxPaymentY2",
        "result_maxPaymentY3", "result_reConfirmMessage",
    ]
    # keep only columns that exist in df_golden
    context_cols      = [c for c in context_cols if c in df_golden.columns]
    golden_label_cols = [golden_col(f) for f in SCORE_FIELDS]

    df_mismatch = (
        df_golden[mismatch_mask][context_cols + golden_label_cols]
        .copy()
        .reset_index(drop=True)
    )
    df_pred_mis = df_pred[mismatch_mask.values].reset_index(drop=True)
    for f in SCORE_FIELDS:
        df_mismatch[f"pred_{f}"] = df_pred_mis[f].values

    mismatch_json = df_mismatch.to_json(orient="records", force_ascii=False, indent=2)
    print(f"\n  {mismatch_mask.sum()} mismatched rows → OptPrompt.")

    # ── 6. Get improved prompt ─────────────────────────────────────────────
    opt_filled = fill_opt_prompt(OptPrompt, current_prompt, mismatch_json)
    print("  Requesting improved prompt…")
    opt_resp       = call_webhook(opt_filled)
    current_prompt = extract_prompt_text(opt_resp)
    print(f"  New prompt ({len(current_prompt)} chars) ready for round {round_idx + 1}.")

print(f"\nOptimisation complete — {len(all_rounds)} round(s) executed.")

## Results Summary

In [ ]:
summary_rows = []
for r in all_rounds:
    row = {"round": r["round"], "prompt_name": r["prompt_name"]}
    row.update({f: f'{v:.1%}' for f, v in r["mismatch_rates"].items()})
    summary_rows.append(row)

df_summary = pd.DataFrame(summary_rows)
print("=" * 60)
print("OPTIMISATION SUMMARY — mismatch rate per field")
print("=" * 60)
display(df_summary)

best = min(
    all_rounds,
    key=lambda r: max(
        (v for v in r["mismatch_rates"].values() if not pd.isna(v)),
        default=float("inf"),
    ),
)
print(f"\nBest round : {best['prompt_name']}")
print(
    f"Max mismatch: "
    f"{max(v for v in best['mismatch_rates'].values() if not pd.isna(v)):.1%}"
)
print("\nAccess results:")
print("  all_rounds[N]['df_test_result']  — labelled DataFrame for round N")
print("  all_rounds[N]['prompt']          — prompt text used in round N")
print("  all_rounds[-1]['prompt']         — final (best last) prompt")